# A sliding-window Dataset for forecasting

Forecasting models train on (past window, future values) pairs cut from one long series. A `torch.utils.data.Dataset` makes that slicing lazy: nothing is copied until an item is asked for.

## The Dataset
Write `WindowDataset(Dataset)` constructed from `(series, window, horizon, stride)`:

- `series`: float64 tensor `(L,)` or `(L, F)`.
- Item `i` starts at `t = i * stride` and returns the pair
  - `x = series[t : t + window]`,
  - `y = series[t + window : t + window + horizon]`.
- Only complete pairs count, so `len(ds)` is $\max\big(0,\ \lfloor (L - \text{window} - \text{horizon}) / \text{stride} \rfloor + 1\big)$.
- Negative indices work like Python lists (`ds[-1]` is the last item). An index outside `[-len, len)` raises `IndexError`.

## The function that's graded
Write `window_items(series, window, horizon, stride, idxs)`:

- `window`, `horizon`, `stride` are ints $\ge 1$; `idxs` is a non-empty list of valid indices.
- Return `(n, X, Y)`: `n = len(ds)`, and `X`, `Y` the items at `idxs` stacked along a new first axis: `X` is `(len(idxs), window)` or `(len(idxs), window, F)`, and `Y` likewise with `horizon`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
from torch.utils.data import Dataset

class WindowDataset(Dataset):
    def __init__(self, series, window, horizon, stride):
        self.series, self.window, self.horizon, self.stride = series, window, horizon, stride

    def __len__(self):
        return len(self.series) // self.stride

    def __getitem__(self, i):
        t = i * self.stride
        return self.series[t:t + self.window], self.series[t + self.window:t + self.window + self.horizon]

def window_items(series, window, horizon, stride, idxs):
    """Return (len(ds), X stacked, Y stacked)."""
    ds = WindowDataset(series, window, horizon, stride)
    xs, ys = zip(*(ds[i] for i in idxs))
    return len(ds), torch.stack(xs), torch.stack(ys)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0], "dtype": "float64"}, 3, 1, 1, [0, 2]], "e": {"__tup__": [3, {"__nd__": [[0.0, 1.0, 2.0], [2.0, 3.0, 4.0]], "dtype": "float64"}, {"__nd__": [[3.0], [5.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0], "dtype": "float64"}, 4, 2, 2, [0, -1]], "e": {"__tup__": [3, {"__nd__": [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]], "dtype": "float64"}, {"__nd__": [[4.0, 5.0], [8.0, 9.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], "dtype": "float64"}, 2, 3, 3, [1, -2]], "e": {"__tup__": [2, {"__nd__": [[3.0, 4.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[5.0, 6.0, 7.0], [2.0, 3.0, 4.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.0, 1.0], [2.0, 3.0], [4.0, 5.0], [6.0, 7.0], [8.0, 9.0], [10.0, 11.0], [12.0, 13.0], [14.0, 15.0], [16.0, 17.0], [18.0, 19.0], [20.0, 21.0], [22.0, 23.0]], "dtype": "float64"}, 3, 2, 2, [0, 1, 3]], "e": {"__tup__": [4, {"__nd__": [[[0.0, 1.0], [2.0, 3.0], [4.0, 5.0]], [[4.0, 5.0], [6.0, 7.0], [8.0, 9.0]], [[12.0, 13.0], [14.0, 15.0], [16.0, 17.0]]], "dtype": "float64"}, {"__nd__": [[[6.0, 7.0], [8.0, 9.0]], [[10.0, 11.0], [12.0, 13.0]], [[18.0, 19.0], [20.0, 21.0]]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0], "dtype": "float64"}, 4, 1, 1, [0, -1]], "e": {"__tup__": [1, {"__nd__": [[0.0, 1.0, 2.0, 3.0], [0.0, 1.0, 2.0, 3.0]], "dtype": "float64"}, {"__nd__": [[4.0], [4.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0], "dtype": "float64"}, 2, 2, 4, [1, 0, -2]], "e": {"__tup__": [2, {"__nd__": [[4.0, 5.0], [0.0, 1.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [[6.0, 7.0], [2.0, 3.0], [2.0, 3.0]], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.18905338179353307, -0.5227484414807474, -0.41306354339189344], [-2.4414673826398556, 1.799707382720902, 1.1441658720372287], [-0.32542283686782436, 0.7738065867276614, 0.28121066979764925], [-0.5538228364240524, 0.9775674511260357, -0.31055654665915255], [-0.3288239040579627, -0.7921467553588982, 0.45495807124085547], [-0.09919805171738795, 0.5452887139646817, -0.6071856998706371], [0.12682784711186987, -0.8922740434297903, 0.8414649723701431], [0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686], [-1.6384425032355252, -1.7294114671544816, -1.50483141386432], [0.8414588934539998, 0.12871565747406846, 1.078342440739298], [0.722430872307499, 0.21057181237528058, 0.28403814525037085], [-0.16976049772313542, 0.8684602112115102, -1.1297159617807548], [-0.4218588261783162, 0.2429388530987352, 1.8014208584493328], [-0.7644641157203993, -1.0790604591369424, -0.5632871985040379], [0.9692722011381628, -0.23500550689182578, 1.324347019236957], [-1.872529517606024, 1.128523141900478, 1.0348662399295387], [-1.4186656898014849, 0.15358188653595775, 1.2157576351945363], [0.08791904230952238, 0.99970142215178, 2.3748869133171437], [0.2739322545923299, -0.2803823139429379, -0.7710521598195307], [0.6480646015444852, -0.19673006635772372, -0.17874637079406783], [-0.1052597784150319, 0.6498696525854939, -1.0663396121649127], [-1.5298764906705564, -2.4338766971489387, 1.1986708993900783], [0.07379335958326627, 1.5101194131223459, -0.00895629811039896], [-0.7421554551991772, 0.477929298799656, -0.07658841293719656], [-1.2541866619736308, -0.885066581450772, 1.766779317484588], [0.3543505953235256, 0.4163865350301778, -0.27655171832101155], [-0.6897195637660919, 0.8916558155709342, -0.10462645267369168], [-0.7591081117479301, -0.13412851934954786, -0.9056060167223513], [0.19004583859111193, 1.12922763006904, -0.8360851661432465]], "dtype": "float64"}, 7, 3, 5, [-1, 0, 2, 4]], "e": {"__tup__": [5, {"__nd__": [[[0.6480646015444852, -0.19673006635772372, -0.17874637079406783], [-0.1052597784150319, 0.6498696525854939, -1.0663396121649127], [-1.5298764906705564, -2.4338766971489387, 1.1986708993900783], [0.07379335958326627, 1.5101194131223459, -0.00895629811039896], [-0.7421554551991772, 0.477929298799656, -0.07658841293719656], [-1.2541866619736308, -0.885066581450772, 1.766779317484588], [0.3543505953235256, 0.4163865350301778, -0.27655171832101155]], [[0.18905338179353307, -0.5227484414807474, -0.41306354339189344], [-2.4414673826398556, 1.799707382720902, 1.1441658720372287], [-0.32542283686782436, 0.7738065867276614, 0.28121066979764925], [-0.5538228364240524, 0.9775674511260357, -0.31055654665915255], [-0.3288239040579627, -0.7921467553588982, 0.45495807124085547], [-0.09919805171738795, 0.5452887139646817, -0.6071856998706371], [0.12682784711186987, -0.8922740434297903, 0.8414649723701431]], [[0.8414588934539998, 0.12871565747406846, 1.078342440739298], [0.722430872307499, 0.21057181237528058, 0.28403814525037085], [-0.16976049772313542, 0.8684602112115102, -1.1297159617807548], [-0.4218588261783162, 0.2429388530987352, 1.8014208584493328], [-0.7644641157203993, -1.0790604591369424, -0.5632871985040379], [0.9692722011381628, -0.23500550689182578, 1.324347019236957], [-1.872529517606024, 1.128523141900478, 1.0348662399295387]], [[0.6480646015444852, -0.19673006635772372, -0.17874637079406783], [-0.1052597784150319, 0.6498696525854939, -1.0663396121649127], [-1.5298764906705564, -2.4338766971489387, 1.1986708993900783], [0.07379335958326627, 1.5101194131223459, -0.00895629811039896], [-0.7421554551991772, 0.477929298799656, -0.07658841293719656], [-1.2541866619736308, -0.885066581450772, 1.766779317484588], [0.3543505953235256, 0.4163865350301778, -0.27655171832101155]]], "dtype": "float64"}, {"__nd__": [[[-0.6897195637660919, 0.8916558155709342, -0.10462645267369168], [-0.7591081117479301, -0.13412851934954786, -0.9056060167223513], [0.19004583859111193, 1.12922763006904, -0.8360851661432465]], [[0.18803508698068597, 0.33057100813532614, 0.41050391297026284], [-1.0107575001533344, 0.7831809961440773, 2.0567028183423686], [-1.6384425032355252, -1.7294114671544816, -1.50483141386432]], [[-1.4186656898014849, 0.15358188653595775, 1.2157576351945363], [0.08791904230952238, 0.99970142215178, 2.3748869133171437], [0.2739322545923299, -0.2803823139429379, -0.7710521598195307]], [[-0.6897195637660919, 0.8916558155709342, -0.10462645267369168], [-0.7591081117479301, -0.13412851934954786, -0.9056060167223513], [0.19004583859111193, 1.12922763006904, -0.8360851661432465]]], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(window_items, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")